# Semantic Search Engine with FAISS + OpenAI Embeddings

## Project objective

In this project we will build a small semantic search engine from scratch using:

- **Python**
- **OpenAI Embeddings API**
- **FAISS**
- **NumPy**

The system will compare two retrieval approaches:

1. **Semantic search** — convert documents and the query into embedding vectors, then search the vectors with a FAISS index.
2. **Keyword search** — use simple word-overlap scoring as a transparent baseline.

We will:

- create a corpus of 60 short documents;
- embed every document with `text-embedding-3-small`;
- store all vectors as a NumPy `float32` array;
- build a FAISS `IndexFlatL2` index;
- verify that all 60 documents were added;
- implement `semantic_search(query, top_k)`;
- implement `keyword_search(query, corpus, top_k)`;
- run 10 test queries through both systems;
- compare the results side by side;
- identify semantic-search wins caused by vocabulary mismatch;
- identify cases where keyword search is more precise;
- discuss cost, latency, indexing, and production trade-offs.

> **Important:** the embedding cells make real API requests. Do not hard-code your API key in the notebook.

# 1. Why do we need a vector index?

A keyword search system mainly asks:

> "How many words from the query also occur in this document?"

Semantic search asks a different question:

> "How close are the meanings of the query and document in vector space?"

The OpenAI `text-embedding-3-small` model converts text into numerical vectors that can be used for relatedness, search, clustering, and retrieval tasks. The current documented price is **$0.02 per 1 million input tokens**. citeturn0search0

FAISS is useful because it provides efficient nearest-neighbour search over vectors. In this project we deliberately use `IndexFlatL2`, which performs an exact flat search so that the mechanics are easy to understand before moving to approximate indexes used at larger scale.

# 2. Install the required packages

If the imports later fail, run this cell once.

```python
%pip install openai faiss-cpu numpy pandas
```

### What does `%pip install` mean?

- `%pip` is a Jupyter/IPython command.
- `install` tells Python's package manager to install a package.
- `openai` gives us the OpenAI Python SDK.
- `faiss-cpu` gives us FAISS running on the CPU.
- `numpy` provides numerical arrays.
- `pandas` makes the final comparison tables easier to read.

The cell is commented out so that the notebook does not reinstall packages every time you run it.

In [ ]:
# %pip install openai faiss-cpu numpy pandas

# 3. Import the libraries

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import faiss

from openai import OpenAI

## New imports

### `os`

Used to read the API key from an environment variable.

### `time`

Used to measure how long embedding generation and search take.

### `numpy`

NumPy gives us the `ndarray` structure used to store the embedding matrix.

### `pandas`

Pandas lets us display the 10-query side-by-side comparison as a clean table.

### `faiss`

FAISS stands for **Facebook AI Similarity Search**. It provides vector indexes and nearest-neighbour search operations.

### `OpenAI`

This is the official Python SDK client used to call the embeddings API.

# 4. Configure the OpenAI client

The API key should be stored outside the notebook.

For example, in a terminal:

```text
export OPENAI_API_KEY="your_api_key_here"
```

On Windows PowerShell:

```text
$env:OPENAI_API_KEY="your_api_key_here"
```

Then Python can read it with `os.getenv()`.

In [ ]:
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY was not found. "
        "Set the environment variable before running the API cells."
    )

client = OpenAI(api_key=api_key)

print("OpenAI client created successfully.")

## New lines explained

```python
api_key = os.getenv("OPENAI_API_KEY")
```

`os.getenv()` reads an environment variable.

```python
if not api_key:
```

Checks whether the key is missing or empty.

```python
raise RuntimeError(...)
```

Stops execution with a clear error message instead of allowing a confusing API error later.

```python
client = OpenAI(api_key=api_key)
```

Creates the object that will communicate with the OpenAI API.

# 5. Create the 60-document corpus

We use six topics with ten short documents each:

1. Technology
2. Cybersecurity
3. Health & Fitness
4. Cooking
5. Travel
6. Finance

The documents are intentionally written with related concepts expressed using different vocabulary. That gives us a useful test for semantic retrieval.

In [ ]:
corpus = [
    # Technology — 0 to 9
    "Cloud platforms let companies run applications on remote computing infrastructure.",
    "Virtual machines allow several operating systems to share one physical server.",
    "Software containers package applications together with their required libraries.",
    "A database stores structured information so applications can retrieve it efficiently.",
    "Caching keeps frequently requested data closer to users and reduces response time.",
    "Load balancers distribute incoming requests across multiple application servers.",
    "An API allows one software system to communicate with another through defined interfaces.",
    "Version control records changes to source code and helps developers collaborate safely.",
    "Continuous integration automatically tests new code before it is merged into a project.",
    "A message queue lets independent services exchange work asynchronously.",

    # Cybersecurity — 10 to 19
    "Multi-factor authentication adds an extra verification step when a user signs in.",
    "Strong unique passwords reduce the risk of unauthorized account access.",
    "Phishing attacks trick people into revealing passwords or sensitive information.",
    "Encryption protects information by transforming readable data into protected ciphertext.",
    "Security patches fix known weaknesses in operating systems and applications.",
    "A firewall controls which network connections are allowed to enter or leave a system.",
    "Least privilege gives each user only the permissions required for their job.",
    "Backups help organizations recover files after ransomware or other destructive incidents.",
    "Security monitoring can detect unusual login behavior and suspicious network activity.",
    "Password managers store credentials securely and can generate strong random passwords.",

    # Health & Fitness — 20 to 29
    "Regular aerobic exercise can improve cardiovascular fitness and stamina.",
    "Strength training helps build muscle and improve physical power.",
    "Adequate sleep supports recovery, concentration, and overall physical performance.",
    "Drinking enough water helps the body maintain normal hydration during exercise.",
    "A balanced diet provides carbohydrates, protein, fats, vitamins, and minerals.",
    "Stretching and mobility work can help maintain comfortable movement around the joints.",
    "Rest days give muscles time to recover after demanding workouts.",
    "Gradually increasing training load can reduce the risk of overuse injuries.",
    "A warm-up prepares the body for exercise by gradually increasing activity.",
    "Tracking heart rate can help athletes understand exercise intensity.",

    # Cooking — 30 to 39
    "Roasting vegetables at high heat can create browned edges and deeper flavor.",
    "A sharp knife makes chopping ingredients faster and more controlled.",
    "Yeast produces gas that helps bread dough rise during fermentation.",
    "Simmering a sauce slowly can concentrate its flavor without excessive boiling.",
    "Salt can enhance the natural taste of many ingredients when used carefully.",
    "Fresh herbs can add aroma and brightness to soups, salads, and pasta.",
    "Resting cooked meat allows its juices to redistribute before slicing.",
    "A kitchen thermometer helps cooks check whether food has reached a safe temperature.",
    "Emulsification combines liquids such as oil and vinegar into a more stable mixture.",
    "Preheating an oven helps baked food cook at the intended temperature from the beginning.",

    # Travel — 40 to 49
    "Checking public transport schedules can make getting around a new city easier.",
    "Travel insurance can help cover certain unexpected medical or trip-related expenses.",
    "A lightweight backpack is useful for walking through cities and hiking trails.",
    "Booking accommodation near a train station can simplify transportation during a trip.",
    "Learning a few local phrases can make everyday communication easier for visitors.",
    "Travelers should check passport and visa requirements before an international journey.",
    "Carrying a reusable water bottle is convenient during long sightseeing days.",
    "Offline maps can help travelers navigate when mobile internet is unavailable.",
    "Visiting museums and historic neighborhoods can provide insight into local culture.",
    "Checking weather forecasts helps travelers choose appropriate clothing and activities.",

    # Finance — 50 to 59
    "A budget helps households plan spending and avoid unnecessary financial stress.",
    "An emergency fund provides savings that can be used for unexpected expenses.",
    "Diversification spreads investments across different assets to reduce concentration risk.",
    "Compound interest allows savings to grow as interest is earned on previous interest.",
    "A credit score can affect the interest rate offered on some types of borrowing.",
    "Index funds track a market index and generally hold a broad collection of securities.",
    "Inflation reduces the purchasing power of money when prices rise over time.",
    "A mortgage is a loan commonly used to finance the purchase of property.",
    "Comparing loan interest rates can help borrowers choose less expensive financing.",
    "Retirement contributions can help people build long-term savings for later life.",
]

topics = (
    ["Technology"] * 10
    + ["Cybersecurity"] * 10
    + ["Health & Fitness"] * 10
    + ["Cooking"] * 10
    + ["Travel"] * 10
    + ["Finance"] * 10
)

document_ids = [f"D{i:02d}" for i in range(len(corpus))]

print("Number of documents:", len(corpus))
print("Number of topic labels:", len(topics))
print("Number of document IDs:", len(document_ids))

## Why use a list?

The corpus is stored as a normal Python list because that is exactly what the assignment asks for.

The position of each document is important:

```text
corpus[0]  ↔ D00 ↔ embedding[0] ↔ FAISS vector 0
corpus[1]  ↔ D01 ↔ embedding[1] ↔ FAISS vector 1
...
```

This alignment lets us recover the original document after FAISS returns a vector position.

In [ ]:
assert len(corpus) >= 50
assert len(corpus) == len(topics) == len(document_ids)

for doc_id, topic, text in zip(document_ids, topics, corpus):
    print(f"{doc_id} | {topic:16} | {text}")

### New command: `assert`

`assert condition` checks that a condition is true.

If the condition is false, Python raises an error.

Here it protects us from accidentally building an index with a different number of documents and labels.

# 6. Choose the embedding model

We use the model required by the assignment:

```text
text-embedding-3-small
```

OpenAI's current model documentation lists this model as an embedding model for relatedness, search, clustering, recommendations, and classification, with a current input price of $0.02 per 1M tokens. citeturn0search0

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"

print("Embedding model:", EMBEDDING_MODEL)

# 7. Embed all 60 documents

We send the whole list to the embeddings endpoint in one request.

We also measure the elapsed time because the assignment asks us to discuss latency and production cost.

In [ ]:
start_time = time.perf_counter()

embedding_response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=corpus
)

embedding_time = time.perf_counter() - start_time

document_vectors = np.array(
    [item.embedding for item in embedding_response.data],
    dtype=np.float32
)

print("Embedding generation complete.")
print("Embedding time:", round(embedding_time, 4), "seconds")
print("Vector matrix shape:", document_vectors.shape)
print("NumPy dtype:", document_vectors.dtype)

## New commands explained

### `time.perf_counter()`

Returns a high-resolution timer.

We calculate:

```python
end_time - start_time
```

to measure elapsed time.

### `client.embeddings.create(...)`

This sends the documents to OpenAI's embeddings endpoint.

### `input=corpus`

The entire Python list is the input batch.

### `np.array(..., dtype=np.float32)`

The API gives us Python lists of floating-point numbers.

We convert them into a NumPy array and explicitly request `float32`.

FAISS works naturally with `float32` vectors, and using 32-bit floats also reduces memory compared with 64-bit floats.

In [ ]:
assert document_vectors.shape[0] == len(corpus)
assert document_vectors.dtype == np.float32

print("All document vectors are present and stored as float32.")

# 8. Inspect the vector dimensions

The matrix has the shape:

```text
(number of documents, embedding dimensions)
```

For example:

```text
(60, D)
```

means 60 documents and D numbers per embedding.

The exact dimensionality is obtained from the live API response rather than hard-coded.

In [ ]:
num_documents, embedding_dimension = document_vectors.shape

print("Documents:", num_documents)
print("Embedding dimensions:", embedding_dimension)
print("First 8 values of D00:", document_vectors[0][:8])

# 9. Build a FAISS `IndexFlatL2`

FAISS indexes need to know the vector dimensionality.

`IndexFlatL2` means:

- **Index** → a searchable collection of vectors.
- **Flat** → every stored vector is kept directly.
- **L2** → Euclidean distance is used.

This is an exact search index. For every query, FAISS compares the query against all stored vectors.

That is simple and excellent for learning. Production systems with very large corpora may use approximate-nearest-neighbour indexes to trade a small amount of recall for speed and memory efficiency.

In [ ]:
index = faiss.IndexFlatL2(embedding_dimension)

print("FAISS index created.")
print("Vectors stored before adding documents:", index.ntotal)

### New command: `faiss.IndexFlatL2()`

```python
faiss.IndexFlatL2(dimension)
```

creates an empty FAISS index whose vectors must contain exactly `dimension` values.

At this point, no documents have been inserted, so:

```python
index.ntotal
```

should be `0`.

# 10. Normalize the vectors before adding them

OpenAI's embedding outputs are L2-normalized to length 1 by default. OpenAI's current FAQ notes that for normalized embeddings, cosine similarity and Euclidean distance produce identical rankings. citeturn0search3

We normalize again explicitly so the notebook makes the assumption visible and robust.

In [ ]:
faiss.normalize_L2(document_vectors)

vector_norms = np.linalg.norm(
    document_vectors,
    axis=1
)

print("Minimum vector norm:", vector_norms.min())
print("Maximum vector norm:", vector_norms.max())

### New command: `faiss.normalize_L2()`

This changes each vector so its Euclidean length becomes approximately `1`.

For a normalized vector:

```text
||v|| ≈ 1
```

This is useful because with unit-length vectors:

```text
L2 distance = 2 - 2 × cosine_similarity
```

Therefore:

- smaller FAISS L2 distance = more similar;
- larger cosine similarity = more similar.

The assignment specifically asks for FAISS distance scores, so we keep the FAISS L2 distances in our results.

# 11. Add all vectors to FAISS

In [ ]:
index.add(document_vectors)

print("Vectors stored in FAISS:", index.ntotal)
print("Expected vectors:", len(corpus))

assert index.ntotal == len(corpus)

print("Verification passed: every document vector was added.")

## Why is `index.ntotal` important?

FAISS does not automatically know how many documents we intended to store.

If we expected 60 documents but `index.ntotal` were 59, one document would be missing.

The assertion therefore verifies:

```text
number of vectors in FAISS
=
number of documents in corpus
```

# 12. Create a helper for embedding one query

Documents are already embedded and stored.

A new search query still needs to be converted into the **same embedding space** before FAISS can compare it with the stored vectors.

In [ ]:
def embed_query(query):
    """Convert one query string into a normalized float32 vector."""

    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a non-empty string.")

    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=[query]
    )

    vector = np.array(
        response.data[0].embedding,
        dtype=np.float32
    ).reshape(1, -1)

    faiss.normalize_L2(vector)

    return vector

## New function concepts

### `def`

Starts a reusable function.

```python
def embed_query(query):
```

means the function accepts one argument called `query`.

### `return`

Sends the final vector back to the code that called the function.

### `.reshape(1, -1)`

FAISS expects a two-dimensional array:

```text
(number_of_queries, vector_dimension)
```

One query therefore has shape:

```text
(1, embedding_dimension)
```

### Why normalize the query?

The stored documents were normalized, so the query should use the same representation before distance calculation.

# 13. Build `semantic_search(query, top_k)`

This is the main semantic retrieval function.

Process:

```text
query
  ↓
OpenAI embedding
  ↓
float32 vector
  ↓
L2 normalization
  ↓
FAISS search
  ↓
nearest vector IDs + distances
  ↓
original corpus documents
```

In [ ]:
def semantic_search(query, top_k=3):
    """
    Search the FAISS index using an embedded query.

    Returns a list of dictionaries containing:
    - rank
    - document ID
    - topic
    - document text
    - FAISS L2 distance
    """

    if top_k < 1:
        raise ValueError("top_k must be at least 1.")

    top_k = min(top_k, len(corpus))

    query_vector = embed_query(query)

    distances, indices = index.search(
        query_vector,
        top_k
    )

    results = []

    for rank, (distance, index_position) in enumerate(
        zip(distances[0], indices[0]),
        start=1
    ):
        results.append({
            "rank": rank,
            "document_id": document_ids[index_position],
            "topic": topics[index_position],
            "document": corpus[index_position],
            "distance": float(distance),
        })

    return results

## Important new FAISS command

```python
distances, indices = index.search(query_vector, top_k)
```

FAISS returns two arrays:

### `distances`

The L2 distance from the query to each nearest vector.

Lower is better.

### `indices`

The positions of the matching vectors inside the FAISS index.

Because we added documents in the same order as `corpus`, an index position maps directly back to:

```python
document_ids[index_position]
corpus[index_position]
topics[index_position]
```

# 14. Test semantic search once

The query below uses different vocabulary from the document about multi-factor authentication.

The goal is to see whether semantic retrieval can connect the intent even when exact words do not match.

In [ ]:
test_query = "How can I make my login safer with an additional verification step?"

semantic_results = semantic_search(
    test_query,
    top_k=5
)

print("Query:", test_query)
print()

for result in semantic_results:
    print(
        f"{result['rank']}. "
        f"{result['document_id']} | "
        f"distance={result['distance']:.4f} | "
        f"{result['document']}"
    )

# 15. Build the keyword-search baseline

Now we deliberately create a much simpler search method.

The keyword score will be:

```text
number of query words appearing in the document
------------------------------------------------
number of query words
```

This is **not** a production search engine. It is a transparent baseline for demonstrating the difference between exact vocabulary overlap and semantic matching.

In [ ]:
def tokenize_simple(text):
    """Convert text into lowercase word tokens."""

    cleaned = (
        text.lower()
        .replace(".", " ")
        .replace(",", " ")
        .replace("?", " ")
        .replace("!", " ")
        .replace(":", " ")
        .replace(";", " ")
        .replace("(", " ")
        .replace(")", " ")
        .replace("-", " ")
    )

    return set(cleaned.split())

## New function: `tokenize_simple()`

We:

1. convert text to lowercase;
2. replace common punctuation with spaces;
3. call `.split()` to separate words;
4. convert the list to a `set`.

A set removes duplicate words.

For example:

```text
"Secure secure account"
```

becomes approximately:

```text
{"secure", "account"}
```

In [ ]:
def keyword_search(query, corpus, top_k=3):
    """
    Rank documents by simple word-overlap score.

    The score is:
        shared query words / total unique query words
    """

    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a non-empty string.")

    if top_k < 1:
        raise ValueError("top_k must be at least 1.")

    query_words = tokenize_simple(query)

    if not query_words:
        raise ValueError("query does not contain searchable words.")

    scored_documents = []

    for position, document in enumerate(corpus):
        document_words = tokenize_simple(document)

        overlap = query_words.intersection(document_words)

        score = len(overlap) / len(query_words)

        scored_documents.append({
            "document_id": document_ids[position],
            "topic": topics[position],
            "document": document,
            "keyword_score": score,
            "overlap_words": sorted(overlap),
        })

    scored_documents.sort(
        key=lambda item: item["keyword_score"],
        reverse=True
    )

    results = []

    for rank, item in enumerate(
        scored_documents[:top_k],
        start=1
    ):
        item = item.copy()
        item["rank"] = rank
        results.append(item)

    return results

## New command: `.intersection()`

```python
query_words.intersection(document_words)
```

finds words that exist in **both** sets.

For example:

```text
query = {"account", "security", "login"}
document = {"account", "security", "password"}

intersection = {"account", "security"}
```

The keyword score is therefore:

```text
2 / 3 = 0.667
```

# 16. Create ten test queries

We deliberately include several kinds of queries:

- vocabulary mismatch, where semantic search should have an advantage;
- exact terminology, where keyword matching can be very precise;
- broad queries;
- ambiguous queries;
- queries with specific technical terms.

The `expected_document` field gives us a human-defined target for evaluating the top result.

In [ ]:
test_queries = [
    {
        "query": "What can I do to make my account sign-in safer with another verification step?",
        "expected_document": "D10",
        "expected_reason": "Vocabulary mismatch: extra verification step means multi-factor authentication.",
    },
    {
        "query": "How can I prepare my body for a workout before the main exercise?",
        "expected_document": "D28",
        "expected_reason": "Vocabulary mismatch: preparing the body before exercise refers to a warm-up.",
    },
    {
        "query": "What should I check before going overseas so I can legally enter another country?",
        "expected_document": "D45",
        "expected_reason": "Vocabulary mismatch: legal entry requirements correspond to passport and visa requirements.",
    },
    {
        "query": "What protects data by turning readable information into ciphertext?",
        "expected_document": "D13",
        "expected_reason": "Exact cybersecurity terminology should favor keyword overlap.",
    },
    {
        "query": "Which document discusses passwords that are unique and hard to guess?",
        "expected_document": "D11",
        "expected_reason": "The distinctive words strongly match the password document.",
    },
    {
        "query": "Which document explains spreading investments across different assets?",
        "expected_document": "D52",
        "expected_reason": "The phrase closely matches diversification terminology.",
    },
    {
        "query": "How can I get around a city when I do not have internet access?",
        "expected_document": "D47",
        "expected_reason": "Semantic search should connect navigation without internet to offline maps.",
    },
    {
        "query": "What helps bread become larger during fermentation?",
        "expected_document": "D32",
        "expected_reason": "The concept of yeast producing gas during fermentation is semantically specific.",
    },
    {
        "query": "What kind of financial savings should I keep for surprise expenses?",
        "expected_document": "D51",
        "expected_reason": "Emergency fund is the intended concept despite different wording.",
    },
    {
        "query": "How can a company spread incoming web requests across several servers?",
        "expected_document": "D05",
        "expected_reason": "The networking concept corresponds directly to load balancing.",
    },
]

print("Number of test queries:", len(test_queries))

# 17. Run all ten queries through both systems

For each query we collect:

- semantic top result;
- semantic distance;
- keyword top result;
- keyword score;
- whether each method matched our expected document.

This gives us an objective table instead of judging results only by looking at them.

In [ ]:
comparison_rows = []

for item in test_queries:
    query = item["query"]
    expected = item["expected_document"]

    semantic_results = semantic_search(query, top_k=3)
    keyword_results = keyword_search(
        query,
        corpus,
        top_k=3
    )

    semantic_top = semantic_results[0]
    keyword_top = keyword_results[0]

    comparison_rows.append({
        "query": query,
        "expected": expected,
        "semantic_top": semantic_top["document_id"],
        "semantic_distance": semantic_top["distance"],
        "semantic_correct": (
            semantic_top["document_id"] == expected
        ),
        "keyword_top": keyword_top["document_id"],
        "keyword_score": keyword_top["keyword_score"],
        "keyword_correct": (
            keyword_top["document_id"] == expected
        ),
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df

### Why use a DataFrame?

A DataFrame is a table-like structure.

It makes the comparison easy to inspect:

```text
query | expected | semantic result | keyword result
```

This directly satisfies the requirement to show the two systems side by side.

# 18. Print the actual top-3 results side by side

The table above compares top-1 results.

This section prints the full top-3 ranking for every query so that differences between the approaches are visible.

In [ ]:
for query_number, item in enumerate(
    test_queries,
    start=1
):
    query = item["query"]

    semantic_results = semantic_search(
        query,
        top_k=3
    )

    keyword_results = keyword_search(
        query,
        corpus,
        top_k=3
    )

    print("=" * 120)
    print(f"QUERY {query_number}: {query}")
    print(f"Expected document: {item['expected_document']}")
    print()

    print("SEMANTIC SEARCH")
    for result in semantic_results:
        print(
            f"  {result['rank']}. "
            f"{result['document_id']} | "
            f"distance={result['distance']:.4f} | "
            f"{result['document']}"
        )

    print()
    print("KEYWORD SEARCH")
    for result in keyword_results:
        print(
            f"  {result['rank']}. "
            f"{result['document_id']} | "
            f"score={result['keyword_score']:.4f} | "
            f"overlap={result['overlap_words']} | "
            f"{result['document']}"
        )

# 19. Identify semantic-search wins

We define a semantic-search win as:

```text
semantic top-1 == expected document
AND
keyword top-1 != expected document
```

This is the strongest type of example for showing vocabulary mismatch.

Because the embedding model is called live, the exact results should always be based on the actual run rather than invented in advance.

In [ ]:
semantic_wins = comparison_df[
    (comparison_df["semantic_correct"] == True)
    & (comparison_df["keyword_correct"] == False)
]

print(
    "Semantic wins found:",
    len(semantic_wins)
)

display(
    semantic_wins[
        [
            "query",
            "expected",
            "semantic_top",
            "semantic_distance",
            "keyword_top",
            "keyword_score",
        ]
    ]
)

## How to document three semantic wins

For the final report, choose three rows from the output above and explain them like this:

> **Query:** [query]  
> **Semantic result:** [document]  
> **Keyword result:** [document]  
> **Diagnosis:** Keyword search failed because the query used different vocabulary from the intended document, while the embedding representation captured the underlying concept.

If fewer than three rows appear, do not invent results. Change the wording of the candidate test queries or add additional vocabulary-mismatch queries and rerun them.

# 20. Identify cases where keyword search is more precise

We define a keyword-precision win as:

```text
keyword top-1 == expected document
AND
semantic top-1 != expected document
```

These cases demonstrate that semantic search is not automatically better for every query.

Exact terminology can be extremely useful when the user knows the precise word they want.

In [ ]:
keyword_wins = comparison_df[
    (comparison_df["keyword_correct"] == True)
    & (comparison_df["semantic_correct"] == False)
]

print(
    "Keyword-search wins found:",
    len(keyword_wins)
)

display(
    keyword_wins[
        [
            "query",
            "expected",
            "semantic_top",
            "semantic_distance",
            "keyword_top",
            "keyword_score",
        ]
    ]
)

## How to document three keyword wins

For the final report, choose three actual rows from this output and explain:

> **Query:** [query]  
> **Keyword result:** [document]  
> **Semantic result:** [document]  
> **Diagnosis:** The query contains distinctive terminology that directly identifies the target document, so exact word overlap is more precise than semantic matching.

Again, use the actual output rather than inventing scores.

# 21. Show the six most useful comparison candidates

Sometimes the live model correctly retrieves the target for both methods.

Those cases are still useful, but they do not satisfy the assignment's "semantic wins" or "keyword wins" definition.

This cell ranks the queries by whether the two systems disagree, so you can quickly find the most interesting cases to discuss.

In [ ]:
disagreement_df = comparison_df[
    comparison_df["semantic_top"]
    != comparison_df["keyword_top"]
].copy()

print(
    "Queries where the two systems chose different top documents:",
    len(disagreement_df)
)

display(disagreement_df)

# 22. Convert FAISS distance to cosine similarity

The assignment asks for FAISS distance scores, so those remain our primary retrieval score.

For normalized vectors:

```text
L2² = 2 - 2 × cosine_similarity
```

Therefore:

```text
cosine_similarity = 1 - (L2_distance / 2)
```

This is useful for interpretation:

- cosine closer to `1` → very similar;
- cosine near `0` → weakly related;
- L2 distance closer to `0` → very similar.

OpenAI's current embeddings FAQ explicitly notes that normalized embedding outputs make cosine and Euclidean distance produce identical rankings. citeturn0search3

In [ ]:
def l2_distance_to_cosine(distance):
    """Convert squared L2 distance for unit vectors to cosine similarity."""
    return 1.0 - (distance / 2.0)


comparison_df["semantic_cosine"] = (
    comparison_df["semantic_distance"]
    .apply(l2_distance_to_cosine)
)

display(
    comparison_df[
        [
            "query",
            "semantic_distance",
            "semantic_cosine",
            "keyword_score",
        ]
    ]
)

### Important detail

FAISS `IndexFlatL2` returns **squared Euclidean distance** values.

That is why the conversion above uses:

```text
cosine = 1 - squared_L2 / 2
```

rather than taking a square root first.

# 23. Measure embedding cost

The embeddings response includes usage information when available.

We use the actual input-token count returned by the API instead of guessing the number of tokens from character counts.

In [ ]:
usage = getattr(
    embedding_response,
    "usage",
    None
)

input_tokens = None

if usage is not None:
    input_tokens = getattr(
        usage,
        "prompt_tokens",
        None
    )

PRICE_PER_MILLION_TOKENS = 0.02

if input_tokens is not None:
    estimated_cost = (
        input_tokens
        / 1_000_000
        * PRICE_PER_MILLION_TOKENS
    )

    print("Corpus embedding input tokens:", input_tokens)
    print(
        f"Estimated corpus embedding cost: "
        f"${estimated_cost:.8f}"
    )
else:
    print(
        "Token usage was not returned by this response, "
        "so exact cost cannot be calculated here."
    )

print(
    "Corpus embedding generation time:",
    round(embedding_time, 4),
    "seconds"
)

## Why production cost has two parts

There are two different embedding costs:

### 1. Indexing cost

Every new or changed document must be embedded.

For a large knowledge base, this can be a significant batch operation.

### 2. Query cost

Every new user query must also be embedded before vector search.

The FAISS lookup itself is local and does not call the OpenAI API.

Therefore the production architecture is usually:

```text
Documents
   ↓
Embedding API
   ↓
FAISS/vector database
   ↓
stored vectors

User query
   ↓
Embedding API
   ↓
query vector
   ↓
FAISS search
   ↓
results
```

The current `text-embedding-3-small` price is $0.02 per 1M input tokens. citeturn0search0

# 24. Measure local FAISS search latency

The following cell measures only the FAISS lookup, not the network/API time required to embed the query.

This distinction matters in production.

In [ ]:
latency_query = "How can I protect an online account from unauthorized access?"

query_vector = embed_query(latency_query)

start_time = time.perf_counter()

distances, indices = index.search(
    query_vector,
    5
)

faiss_latency = time.perf_counter() - start_time

print(
    "Local FAISS search time:",
    round(faiss_latency * 1000, 4),
    "milliseconds"
)

for rank, (distance, position) in enumerate(
    zip(distances[0], indices[0]),
    start=1
):
    print(
        f"{rank}. {document_ids[position]} | "
        f"distance={distance:.4f} | "
        f"{corpus[position]}"
    )

### Why this measurement matters

The end-to-end user experience contains at least two components:

```text
query embedding API latency
+
local vector-search latency
```

For a small local FAISS index, the vector lookup can be extremely fast.

At production scale, however, the system may also include:

- network latency;
- vector database latency;
- reranking;
- metadata filtering;
- document retrieval;
- response generation.

So "FAISS search is fast" does not automatically mean the complete application is low-latency.

# 25. Production trade-offs: semantic vs keyword search

## Semantic search is better when:

- users describe an idea without knowing the exact terminology;
- documents and queries use synonyms;
- users ask natural-language questions;
- the same concept can be expressed in many ways;
- multilingual or paraphrase retrieval is important.

### Example

```text
Query:
"How do I make my login safer with another verification step?"

Document:
"Multi-factor authentication adds an extra verification step when a user signs in."
```

Keyword overlap may be weak because the query does not literally contain "multi-factor authentication".

An embedding model can represent both texts as semantically related.

## Keyword search is better when:

- exact terminology matters;
- identifiers must match exactly;
- product codes, ticket IDs, filenames, or error codes are important;
- users intentionally search for a particular phrase;
- interpretability and deterministic matching are priorities.

For example:

```text
ERROR E1042
```

should not be replaced by a vague semantically related document.

Similarly, a query for a specific API name, chemical name, legal clause, or product SKU may benefit from exact lexical matching.

# 26. Why production systems often combine both

A practical search engine does not have to choose only one method.

A common architecture is:

```text
                 ┌── Keyword search ──┐
User query ──────┤                     ├── Candidate results
                 └── Semantic search ─┘
                            ↓
                       Reranking
                            ↓
                       Final results
```

Keyword retrieval provides precise lexical matching.

Semantic retrieval provides concept matching.

Combining them can reduce the weaknesses of either approach alone.

# 27. Production scaling considerations

## Embedding cost

The current `text-embedding-3-small` price is $0.02 per 1M input tokens. citeturn0search0

For 60 tiny documents, the cost is extremely small.

At millions of documents, however, you should:

- batch indexing work;
- avoid re-embedding unchanged documents;
- cache embeddings;
- track token usage;
- update only changed documents.

## Query latency

Every new query needs an embedding API call.

This introduces network latency.

Possible improvements include:

- keeping the vector index local when appropriate;
- caching repeated query embeddings;
- reducing unnecessary API calls;
- using a lower-dimensional representation when suitable.

## FAISS index size

`IndexFlatL2` stores all vectors directly.

For `N` vectors and `D` dimensions using float32:

```text
approximate vector memory
= N × D × 4 bytes
```

For example, if a model produced 1536-dimensional float32 vectors:

```text
1 vector ≈ 1536 × 4
         ≈ 6144 bytes
         ≈ 6 KB
```

One million such vectors would require roughly 6.1 GB just for the raw vector values, before index and application overhead.

That is why large systems often consider compressed or approximate vector indexes.

# 28. Final results summary

The following cells produce the key submission outputs.

### FAISS verification

In [ ]:
print("========== FAISS VERIFICATION ==========")
print("Corpus documents:", len(corpus))
print("Vector matrix shape:", document_vectors.shape)
print("Vector dtype:", document_vectors.dtype)
print("FAISS index size:", index.ntotal)
print("All vectors stored:", index.ntotal == len(corpus))

### Ten-query side-by-side summary

In [ ]:
summary_columns = [
    "query",
    "expected",
    "semantic_top",
    "semantic_distance",
    "keyword_top",
    "keyword_score",
    "semantic_correct",
    "keyword_correct",
]

display(comparison_df[summary_columns])

### Semantic-search wins

In [ ]:
if len(semantic_wins) > 0:
    display(semantic_wins)
else:
    print(
        "No semantic-win query was found in this run. "
        "Add more synonym/paraphrase queries and rerun the evaluation."
    )

### Keyword-search wins

In [ ]:
if len(keyword_wins) > 0:
    display(keyword_wins)
else:
    print(
        "No keyword-win query was found in this run. "
        "Add more exact-term queries and rerun the evaluation."
    )

# 29. Final written conclusion

## What this experiment demonstrates

A keyword system and a semantic system represent two different ideas of relevance.

### Keyword search

Keyword search rewards literal overlap.

If the query contains words that appear in a document, the document can receive a high score.

This makes keyword search:

- simple;
- cheap;
- explainable;
- deterministic;
- excellent for exact terms.

But it is vulnerable to **vocabulary mismatch**.

### Semantic search

Semantic search converts both documents and queries into vectors.

The system then retrieves vectors that are close to the query vector.

This allows the search engine to connect different wording that expresses a similar idea.

This is why embeddings are especially useful for:

- natural-language questions;
- semantic retrieval;
- recommendations;
- clustering;
- RAG systems.

OpenAI's current documentation specifically lists search and retrieval among embedding use cases. citeturn0search0

### Why FAISS?

FAISS gives us a dedicated vector-search data structure.

Instead of manually calculating the distance between the query and every document in Python, we ask the index to find the nearest vectors.

`IndexFlatL2` is particularly useful for learning because it performs exact nearest-neighbour search.

### The main production lesson

The best retrieval system is often not:

```text
semantic OR keyword
```

but:

```text
semantic + keyword + reranking
```

Each method captures a different form of relevance.

---

## Submission checklist

- [x] At least 50 short documents — this project uses 60.
- [x] OpenAI `text-embedding-3-small`.
- [x] All vectors stored as NumPy `float32`.
- [x] FAISS `IndexFlatL2`.
- [x] FAISS index size printed and verified.
- [x] `semantic_search(query, top_k)` implemented.
- [x] Query embedding performed before FAISS search.
- [x] Distance scores returned.
- [x] `keyword_search(query, corpus, top_k)` implemented.
- [x] Ten queries tested through both systems.
- [x] Side-by-side comparison table.
- [x] Semantic-win analysis.
- [x] Keyword-precision-win analysis.
- [x] Cost discussion.
- [x] Latency discussion.
- [x] Production scaling discussion.
- [x] Explanation of semantic vs keyword retrieval.